In [2]:
! pip install -U bitsandbytes peft datasets matplotlib

In [3]:
!nvidia-smi

import torch
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'Not available'}")

Fri May 30 22:22:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:D5:00.0 Off |                    0 |
|  0%   30C    P8             21W /  300W |       1MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import csv
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict
import os
import numpy as np
from strong_reject.evaluate import evaluate_dataset

after_csv = "../dataset/cautious_eval_output_vllm.csv"

def read_csv_by_repetition(csv_path, target_repetitions=[1, 2, 3, 4, 5]):
    """
    Read CSV and separate data by repetition number.
    Returns dictionaries containing datasets for each repetition.
    """
    # Dictionary to store data for each repetition
    repetition_data = defaultdict(lambda: {
        'prompts': [],
        'responses': [],
        'cot_responses': [],
        'output_responses': []
    })
    
    discarded_counts = defaultdict(int)  # Count discarded rows per repetition
    total_rows = 0
    
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in tqdm(reader, desc="Processing rows"):
            total_rows += 1
            repetition = int(row.get("repetition", 1))  # Default to 1 if no repetition column
            
            # Only process target repetitions
            if repetition not in target_repetitions:
                continue
                
            # Check if response contains the </think> tag
            if "</think>" in row["response"]:
                repetition_data[repetition]['prompts'].append(row["forbidden_prompt"])
                repetition_data[repetition]['responses'].append(row['response'])

                # Split response at </think> tag
                parts = row["response"].split("</think>", 1)  # Split at first occurrence

                # Add chain of thought (everything before </think>)
                repetition_data[repetition]['cot_responses'].append(parts[0].strip())
                # Add output (everything after </think>)
                repetition_data[repetition]['output_responses'].append(parts[1].strip())
            else:
                discarded_counts[repetition] += 1

    # Create datasets for each repetition
    datasets = {}
    cot_datasets = {}
    output_datasets = {}
    
    for rep in target_repetitions:
        if rep in repetition_data:
            data = repetition_data[rep]
            
            # Create main dataset
            datasets[rep] = Dataset.from_dict({
                "forbidden_prompt": data['prompts'],
                "response": data['responses']
            })

            # Create chain of thought dataset
            cot_datasets[rep] = Dataset.from_dict({
                "forbidden_prompt": data['prompts'],
                "response": data['cot_responses']
            })

            # Create output dataset
            output_datasets[rep] = Dataset.from_dict({
                "forbidden_prompt": data['prompts'],
                "response": data['output_responses']
            })
            
            print(f"Repetition {rep}: {len(data['prompts'])} valid responses, {discarded_counts[rep]} discarded")
        else:
            print(f"Warning: No data found for repetition {rep}")
            # Create empty datasets
            datasets[rep] = Dataset.from_dict({"forbidden_prompt": [], "response": []})
            cot_datasets[rep] = Dataset.from_dict({"forbidden_prompt": [], "response": []})
            output_datasets[rep] = Dataset.from_dict({"forbidden_prompt": [], "response": []})

    print(f"Total rows processed: {total_rows}")
    return datasets, cot_datasets, output_datasets

def calculate_strongreject_statistics(output_datasets, target_repetitions=[1, 2, 3, 4, 5]):
    """
    Calculate StrongREJECT scores and compute mean and standard deviation for each repetition.
    """
    statistics = {}
    evaluated_datasets = {}
    
    print("\nEvaluating datasets with StrongREJECT...")
    print("=" * 60)
    
    for rep in target_repetitions:
        if len(output_datasets[rep]) > 0:
            print(f"\nEvaluating Repetition {rep}...")
            
            # Evaluate the dataset
            eval_result = evaluate_dataset(output_datasets[rep], ["strongreject_finetuned"])
            evaluated_datasets[rep] = eval_result
            
            # Extract scores
            scores = eval_result["score"]
            
            # Calculate statistics
            mean_score = np.mean(scores)
            std_score = np.std(scores)
            median_score = np.median(scores)
            min_score = np.min(scores)
            max_score = np.max(scores)
            
            statistics[rep] = {
                'mean': mean_score,
                'std': std_score,
                'median': median_score,
                'min': min_score,
                'max': max_score,
                'count': len(scores),
                'scores': scores
            }
            
            print(f"  Samples evaluated: {len(scores)}")
            print(f"  Mean score: {mean_score:.4f}")
            print(f"  Std deviation: {std_score:.4f}")
        else:
            print(f"\nSkipping Repetition {rep} (no data)")
            statistics[rep] = {
                'mean': 0.0,
                'std': 0.0,
                'median': 0.0,
                'min': 0.0,
                'max': 0.0,
                'count': 0,
                'scores': []
            }
            evaluated_datasets[rep] = None
    
    return statistics, evaluated_datasets

def calculate_combined_statistics(statistics, target_repetitions=[1, 2, 3, 4, 5]):
    """
    Calculate combined mean and standard deviation across all repetitions.
    """
    # Combine all scores from all repetitions
    all_scores = []
    for rep in target_repetitions:
        if statistics[rep]['count'] > 0:
            all_scores.extend(statistics[rep]['scores'])
    
    if len(all_scores) == 0:
        return {
            'combined_mean': 0.0,
            'combined_std': 0.0,
            'combined_count': 0,
            'combined_median': 0.0,
            'combined_min': 0.0,
            'combined_max': 0.0
        }
    
    # Calculate combined statistics
    combined_stats = {
        'combined_mean': np.mean(all_scores),
        'combined_std': np.std(all_scores),
        'combined_count': len(all_scores),
        'combined_median': np.median(all_scores),
        'combined_min': np.min(all_scores),
        'combined_max': np.max(all_scores)
    }
    
    return combined_stats

def print_detailed_statistics(statistics, target_repetitions=[1, 2, 3, 4, 5]):
    """
    Print detailed statistics for all repetitions including combined statistics.
    """
    print("\n" + "="*80)
    print("STRONGREJECT SCORE STATISTICS BY REPETITION")
    print("="*80)
    
    # Individual repetition statistics
    for rep in target_repetitions:
        stats = statistics[rep]
        print(f"\nRepetition {rep}:")
        print("-" * 40)
        print(f"  Sample count:      {stats['count']}")
        print(f"  Mean score:        {stats['mean']:.4f}")
        print(f"  Standard deviation: {stats['std']:.4f}")
        print(f"  Median score:      {stats['median']:.4f}")
        print(f"  Min score:         {stats['min']:.4f}")
        print(f"  Max score:         {stats['max']:.4f}")
    
    # Calculate and display combined statistics
    combined_stats = calculate_combined_statistics(statistics, target_repetitions)
    print(f"\n" + "="*80)
    print("COMBINED STATISTICS ACROSS ALL REPETITIONS")
    print("="*80)
    print(f"  Total samples:           {combined_stats['combined_count']}")
    print(f"  Combined mean:           {combined_stats['combined_mean']:.4f}")
    print(f"  Combined std deviation:  {combined_stats['combined_std']:.4f}")
    print(f"  Combined median:         {combined_stats['combined_median']:.4f}")
    print(f"  Combined min:            {combined_stats['combined_min']:.4f}")
    print(f"  Combined max:            {combined_stats['combined_max']:.4f}")
    
    # Summary comparison
    print(f"\n" + "="*60)
    print("SUMMARY COMPARISON")
    print("="*60)
    
    # Find best and worst performing repetitions
    valid_reps = [rep for rep in target_repetitions if statistics[rep]['count'] > 0]
    if valid_reps:
        means = [statistics[rep]['mean'] for rep in valid_reps]
        best_rep = valid_reps[np.argmax(means)]
        worst_rep = valid_reps[np.argmin(means)]
        
        print(f"Best performing repetition:  Rep {best_rep} (mean: {statistics[best_rep]['mean']:.4f})")
        print(f"Worst performing repetition: Rep {worst_rep} (mean: {statistics[worst_rep]['mean']:.4f})")
        print(f"Score range across reps:     {np.max(means) - np.min(means):.4f}")
        
        # Compare individual rep means to combined mean
        print(f"\nComparison to combined mean ({combined_stats['combined_mean']:.4f}):")
        for rep in target_repetitions:
            if statistics[rep]['count'] > 0:
                diff = statistics[rep]['mean'] - combined_stats['combined_mean']
                print(f"  Rep {rep}: {statistics[rep]['mean']:.4f} ({diff:+.4f} from combined)")
        
        # Show progression
        print(f"\nMean scores by repetition:")
        for rep in target_repetitions:
            if statistics[rep]['count'] > 0:
                print(f"  Rep {rep}: {statistics[rep]['mean']:.4f} ± {statistics[rep]['std']:.4f}")
    
    # Create summary table
    print(f"\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    print(f"{'Rep':<4} {'Count':<6} {'Mean':<8} {'Std Dev':<8} {'Median':<8} {'Min':<6} {'Max':<6}")
    print("-" * 80)
    for rep in target_repetitions:
        stats = statistics[rep]
        print(f"{rep:<4} {stats['count']:<6} {stats['mean']:<8.4f} {stats['std']:<8.4f} "
              f"{stats['median']:<8.4f} {stats['min']:<6.4f} {stats['max']:<6.4f}")
    
    # Add combined row
    print("-" * 80)
    print(f"{'ALL':<4} {combined_stats['combined_count']:<6} {combined_stats['combined_mean']:<8.4f} "
          f"{combined_stats['combined_std']:<8.4f} {combined_stats['combined_median']:<8.4f} "
          f"{combined_stats['combined_min']:<6.4f} {combined_stats['combined_max']:<6.4f}")
    
    return combined_stats

# Main execution
print("Processing CSV file and evaluating with StrongREJECT...")

# Process the CSV for repetitions 1-5
after_datasets, after_cot_datasets, after_op_datasets = read_csv_by_repetition(after_csv, [1, 2, 3, 4, 5])

# Print dataset summary
print("\nDataset Summary:")
for rep in [1, 2, 3, 4, 5]:
    print(f"Repetition {rep}:")
    print(f"  Main dataset: {len(after_datasets[rep])} samples")
    print(f"  CoT dataset: {len(after_cot_datasets[rep])} samples") 
    print(f"  Output dataset: {len(after_op_datasets[rep])} samples")

# Calculate StrongREJECT statistics
strongreject_stats, evaluated_datasets = calculate_strongreject_statistics(after_op_datasets, [1, 2, 3, 4, 5])

# Print detailed statistics
combined_statistics = print_detailed_statistics(strongreject_stats, [1, 2, 3, 4, 5])

# Access combined statistics
print(f"\nQuick access to combined statistics:")
print(f"Combined Mean: {combined_statistics['combined_mean']:.4f}")
print(f"Combined Std Dev: {combined_statistics['combined_std']:.4f}")
print(f"Total Samples: {combined_statistics['combined_count']}")

# Store individual evaluated datasets for further use
rep1_op_eval = evaluated_datasets[1]
rep2_op_eval = evaluated_datasets[2]
rep3_op_eval = evaluated_datasets[3]
rep4_op_eval = evaluated_datasets[4]
rep5_op_eval = evaluated_datasets[5]

# Optional: Save statistics to file including combined stats
def save_statistics_to_csv(statistics, combined_stats, filename="../results/strongreject_statistics.csv"):
    """Save statistics to CSV file for further analysis."""
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    with open(filename, 'w', newline='') as csvfile:
        fieldnames = ['repetition', 'count', 'mean', 'std_dev', 'median', 'min', 'max']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        for rep, stats in statistics.items():
            writer.writerow({
                'repetition': rep,
                'count': stats['count'],
                'mean': stats['mean'],
                'std_dev': stats['std'],
                'median': stats['median'],
                'min': stats['min'],
                'max': stats['max']
            })
        
        # Add combined statistics row
        writer.writerow({
            'repetition': 'COMBINED',
            'count': combined_stats['combined_count'],
            'mean': combined_stats['combined_mean'],
            'std_dev': combined_stats['combined_std'],
            'median': combined_stats['combined_median'],
            'min': combined_stats['combined_min'],
            'max': combined_stats['combined_max']
        })
    
    print(f"\nStatistics saved to: {filename}")

# Uncomment to save statistics including combined stats
# save_statistics_to_csv(strongreject_stats, combined_statistics)

Processing CSV file and evaluating with StrongREJECT...


Processing rows: 580it [00:00, 9217.48it/s]


Repetition 1: 115 valid responses, 1 discarded
Repetition 2: 115 valid responses, 1 discarded
Repetition 3: 115 valid responses, 1 discarded
Repetition 4: 114 valid responses, 2 discarded
Repetition 5: 116 valid responses, 0 discarded
Total rows processed: 580

Dataset Summary:
Repetition 1:
  Main dataset: 115 samples
  CoT dataset: 115 samples
  Output dataset: 115 samples
Repetition 2:
  Main dataset: 115 samples
  CoT dataset: 115 samples
  Output dataset: 115 samples
Repetition 3:
  Main dataset: 115 samples
  CoT dataset: 115 samples
  Output dataset: 115 samples
Repetition 4:
  Main dataset: 114 samples
  CoT dataset: 114 samples
  Output dataset: 114 samples
Repetition 5:
  Main dataset: 116 samples
  CoT dataset: 116 samples
  Output dataset: 116 samples

Evaluating datasets with StrongREJECT...

Evaluating Repetition 1...


Map: 100%|██████████| 115/115 [01:27<00:00,  1.32 examples/s]


  Samples evaluated: 115
  Mean score: 0.1522
  Std deviation: 0.2650

Evaluating Repetition 2...


...

  Samples evaluated: 115
  Mean score: 0.1202
  Std deviation: 0.2362

Evaluating Repetition 4...


Map: 100%|██████████| 114/114 [01:02<00:00,  1.83 examples/s]


  Samples evaluated: 114
  Mean score: 0.1276
  Std deviation: 0.2456

Evaluating Repetition 5...


Map: 100%|██████████| 116/116 [01:03<00:00,  1.82 examples/s]


  Samples evaluated: 116
  Mean score: 0.1672
  Std deviation: 0.2951

STRONGREJECT SCORE STATISTICS BY REPETITION

Repetition 1:
----------------------------------------
  Sample count:      115
  Mean score:        0.1522
  Standard deviation: 0.2650
  Median score:      0.0080
  Min score:         0.0004
  Max score:         0.9852

Repetition 2:
----------------------------------------
  Sample count:      115
  Mean score:        0.1339
  Standard deviation: 0.2690
  Median score:      0.0060
  Min score:         0.0005
  Max score:         0.9387

Repetition 3:
----------------------------------------
  Sample count:      115
  Mean score:        0.1202
  Standard deviation: 0.2362
  Median score:      0.0038
  Min score:         0.0004
  Max score:         0.9917

Repetition 4:
----------------------------------------
  Sample count:      114
  Mean score:        0.1276
  Standard deviation: 0.2456
  Median score:      0.0068
  Min score:         0.0003
  Max score:         0.95